# Phase 5: CrossEncoder Reranking

This notebook:
- Loads the CrossEncoder reranker
- Tests reranking on sample queries
- Compares before/after rankings
- Measures latency

## 5.1 Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import time
from tqdm import tqdm

from src.retrieval import HybridRetriever
from src.reranker import CrossEncoderReranker, load_reranker

print("✓ Imports successful")

✓ Imports successful


## 5.2 Load Components

In [2]:
# Load retriever
retriever = HybridRetriever(indices_dir='../data/indices')
print("✓ Retriever loaded")

Loading BM25 from ../data/indices/bm25_index.pkl...
✓ BM25 loaded: 100,000 documents
Loading FAISS from ../data/indices/faiss_index.bin...
✓ FAISS loaded: 100,000 vectors
Loading embedding model...
✓ HybridRetriever initialized
  Products: 100,000
✓ Retriever loaded


In [3]:
# Load products for display
df_products = pd.read_parquet('../data/processed/products.parquet')
product_lookup = df_products.set_index('product_id').to_dict('index')
print(f"✓ Loaded {len(df_products):,} products")

✓ Loaded 982,641 products


In [4]:
# Load CrossEncoder reranker (default: balanced)
reranker = load_reranker("balanced")

# Other options:
# reranker = load_reranker("fast")     # Faster, slightly lower quality
# reranker = load_reranker("quality")  # Best quality, slower

✓ Loaded CrossEncoder: cross-encoder/ms-marco-MiniLM-L-12-v2


## 5.3 Test Single Query

In [5]:
# Test query
query = "ceramic mugs bulk"

# Get hybrid results
results = retriever.search(query, method="hybrid", top_k=50)
print(f"Query: '{query}'")
print(f"Retrieved {len(results)} candidates")

Query: 'ceramic mugs bulk'
Retrieved 50 candidates


In [6]:
# Add product titles to results
for r in results:
    pid = r['product_id']
    if pid in product_lookup:
        r['product_title'] = product_lookup[pid].get('product_title', '')
    else:
        r['product_title'] = ''

print("✓ Added product titles")

✓ Added product titles


In [7]:
# Rerank
start = time.time()
reranked = reranker.rerank(query, results, top_k=10)
latency = (time.time() - start) * 1000

print(f"Reranking latency: {latency:.0f}ms")

Reranking latency: 295ms


In [8]:
# Show before/after comparison
print(f"\nQuery: '{query}'\n")
print("BEFORE (Hybrid) → AFTER (CrossEncoder)")
print("=" * 70)

for r in reranked:
    title = r['product_title'][:50] if r['product_title'] else 'N/A'
    old_rank = r['original_rank']
    new_rank = r['new_rank']
    move = old_rank - new_rank
    arrow = f"↑{move}" if move > 0 else f"↓{abs(move)}" if move < 0 else "="
    score = r['rerank_score']
    print(f"{new_rank:2d}. (was {old_rank:2d}) {arrow:4s} | {score:.3f} | {title}")


Query: 'ceramic mugs bulk'

BEFORE (Hybrid) → AFTER (CrossEncoder)
 1. (was 22) ↑21  | 3.036 | Mugaholics Stackable Cappuccino Cups Set of 6, 8 o
 2. (was  6) ↑4   | 2.959 | AmorArc Coffee Mugs Set of 6, Large Ceramic Coffee
 3. (was 28) ↑25  | 2.926 | DOWAN 15 oz Coffee Mug Sets, Set of 2 Large Cerami
 4. (was  3) ↓1   | 2.761 | Set of 6 Novelty "Diner" Mugs with Handle - Cerami
 5. (was 32) ↑27  | 2.677 | Glory Haus: Happy & Uplifting - Ceramic Mugs 16 oz
 6. (was 10) ↑4   | 2.262 | LIFVER 18 Ounces Large Coffee Mugs Set of 2, Large
 7. (was 37) ↑30  | 2.127 | Ello Ogden Ceramic Travel Mug with Friction-Fit Li
 8. (was  4) ↓4   | 2.080 | Amici Home Onyx Black/White 20 oz Ceramic Coffee M
 9. (was  1) ↓8   | 1.979 | Serami 15oz White Funnel Ceramic Tall Coffee Mugs 
10. (was 11) ↑1   | 1.942 | Starbucks Travel Mugs Gift Set - 2 Double Wall Cer


## 5.4 Test Multiple Queries

In [9]:
# Test queries (mix of regular and wholesale)
test_queries = [
    "ceramic mugs bulk",
    "organic cotton t-shirts wholesale",
    "bluetooth headphones",
    "stainless steel water bottle case pack",
    "yoga mat",
    "wooden cutting board set",
    "LED desk lamp",
    "leather wallet men"
]

In [10]:
# Run reranking on all queries
latencies = []

for query in tqdm(test_queries, desc="Reranking"):
    # Retrieve
    results = retriever.search(query, method="hybrid", top_k=50)
    
    # Add titles
    for r in results:
        pid = r['product_id']
        if pid in product_lookup:
            r['product_title'] = product_lookup[pid].get('product_title', '')
    
    # Rerank and measure
    start = time.time()
    reranked = reranker.rerank(query, results, top_k=10)
    latencies.append((time.time() - start) * 1000)

print(f"\nLatency Stats:")
print(f"  Mean: {np.mean(latencies):.0f}ms")
print(f"  P50:  {np.percentile(latencies, 50):.0f}ms")
print(f"  P95:  {np.percentile(latencies, 95):.0f}ms")

Reranking: 100%|██████████| 8/8 [00:03<00:00,  2.53it/s]


Latency Stats:
  Mean: 175ms
  P50:  166ms
  P95:  236ms


## 5.5 Compare Before/After for All Queries

In [11]:
# Show comparison for each query
for query in test_queries[:4]:  # First 4 for brevity
    results = retriever.search(query, method="hybrid", top_k=50)
    
    for r in results:
        pid = r['product_id']
        if pid in product_lookup:
            r['product_title'] = product_lookup[pid].get('product_title', '')
    
    comparison = reranker.compare_rankings(query, results, top_k=5)
    
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    print("\nBEFORE (Hybrid):")
    for item in comparison['before']:
        print(f"  {item}")
    print("\nAFTER (CrossEncoder):")
    for item in comparison['after']:
        print(f"  {item}")


Query: 'ceramic mugs bulk'

BEFORE (Hybrid):
  1. Serami 15oz White Funnel Ceramic Tall Coffee Mugs with Large
  2. Amici Home Morganite Pink/White 20 oz Ceramic Coffee Mugs, S
  3. Set of 6 Novelty "Diner" Mugs with Handle - Ceramic, Multico
  4. Amici Home Onyx Black/White 20 oz Ceramic Coffee Mugs, Set o
  5. Cute Marshmallow Shaped Hot Chocolate Mugs-Ceramic-Set of 4

AFTER (CrossEncoder):
  1. Mugaholics Stackable Cappuccino Cups Set of 6, 8 o ↑21
  2. AmorArc Coffee Mugs Set of 6, Large Ceramic Coffee ↑4
  3. DOWAN 15 oz Coffee Mug Sets, Set of 2 Large Cerami ↑25
  4. Set of 6 Novelty "Diner" Mugs with Handle - Cerami ↓1
  5. Glory Haus: Happy & Uplifting - Ceramic Mugs 16 oz ↑27

Query: 'organic cotton t-shirts wholesale'

BEFORE (Hybrid):
  1. Merino.tech 100% Organic Merino Wool Lightweight Men's T-Shi
  2. Calvin Klein Men's Cotton Stretch Multipack Crew Neck T-Shir
  3. Calvin Klein Men's Cotton Stretch Multipack V Neck T-Shirts,
  4. Calvin Klein Men's Cotton Classics Mult

## 5.6 Full Pipeline Function

In [12]:
def search_and_rerank(query, retriever, reranker, product_lookup, top_k=10):
    """
    Full search pipeline: retrieve + rerank.
    
    Args:
        query: Search query
        retriever: HybridRetriever instance
        reranker: CrossEncoderReranker instance
        product_lookup: Dict of product_id -> product info
        top_k: Number of results to return
    
    Returns:
        List of reranked results with product info
    """
    # Retrieve candidates (get more than needed for reranking)
    results = retriever.search(query, method="hybrid", top_k=50)
    
    # Add product info
    for r in results:
        pid = r['product_id']
        if pid in product_lookup:
            r['product_title'] = product_lookup[pid].get('product_title', '')
            r['product_brand'] = product_lookup[pid].get('product_brand', '')
    
    # Rerank
    reranked = reranker.rerank(query, results, top_k=top_k)
    
    return reranked

In [13]:
# Test full pipeline
query = "eco-friendly water bottles wholesale"

start = time.time()
results = search_and_rerank(query, retriever, reranker, product_lookup, top_k=10)
total_time = (time.time() - start) * 1000

print(f"Query: '{query}'")
print(f"Total pipeline time: {total_time:.0f}ms\n")

for r in results:
    title = r.get('product_title', 'N/A')[:55]
    brand = r.get('product_brand', 'N/A')
    score = r['rerank_score']
    print(f"{r['new_rank']:2d}. [{score:.3f}] {title} ({brand})")

Query: 'eco-friendly water bottles wholesale'
Total pipeline time: 387ms

 1. [3.766] Tree Tribe Stainless Steel Water Bottle 20 oz - Indestr (Tree Tribe)
 2. [2.925] Embrava Best Sports Water Bottle - 32oz Large - Fast Fl (Embrava)
 3. [2.875] Super Sparrow Sports Water Bottle12oz&17oz&25oz&32oz&50 (Super Sparrow)
 4. [2.669] Kaelon Dish and Hand Soap Dispenser (Set of 2, 16oz) Ec (Kaelon)
 5. [1.668] 12Pcs 10ml Glass Roll On Bottle with Bamboo Lid for Ess (CREATIEE-PRO)
 6. [1.429] Abitzon NEW Nail Polish Set (10 Bottles) - Non-Toxic Ec (Abitzon)
 7. [-0.082] 500-CT Disposable Gray 12-OZ Hot Beverage Cups with Rip (Restaurantware)
 8. [-1.734] Choary Eco-friendly Unbreakable Reusable Drinking Cup f (choary)
 9. [-2.036] Healthy Human Water Bottles, BPA Free Sports Travel Sta (Healthy Human)
10. [-2.173] H2O Basics BPA-Free Sport Water Bottles 25 oz, Tritan N (H2O Basics)


## 5.7 Summary

In [14]:
print("Phase 5 Complete!")
print("="*50)
print(f"Model: {reranker.model_name}")
print(f"Avg reranking latency: {np.mean(latencies):.0f}ms")
print(f"P95 latency: {np.percentile(latencies, 95):.0f}ms")
print("\nNext: Phase 6 - Evaluation")

Phase 5 Complete!
Model: cross-encoder/ms-marco-MiniLM-L-12-v2
Avg reranking latency: 175ms
P95 latency: 236ms

Next: Phase 6 - Evaluation
